# Workflow Humain dans la Boucle avec Microsoft Agent Framework

## 🎯 Objectifs d'Apprentissage

Dans ce carnet, vous apprendrez à implémenter des workflows **humain dans la boucle** en utilisant le `RequestInfoExecutor` de Microsoft Agent Framework. Ce modèle puissant vous permet de mettre en pause les workflows d'IA pour recueillir des informations humaines, rendant vos agents interactifs et donnant aux humains le contrôle sur des décisions critiques.

## 🔄 Qu'est-ce que le Human-in-the-Loop ?

**Human-in-the-loop (HITL)** est un modèle de conception où les agents IA interrompent leur exécution pour demander une entrée humaine avant de continuer. Ceci est essentiel pour :

- ✅ **Décisions critiques** - Obtenir l'approbation humaine avant de prendre des actions importantes
- ✅ **Situations ambiguës** - Permettre aux humains de clarifier quand l'IA est incertaine
- ✅ **Préférences utilisateurs** - Demander aux utilisateurs de choisir entre plusieurs options
- ✅ **Conformité & sécurité** - Assurer la supervision humaine pour les opérations réglementées
- ✅ **Expériences interactives** - Construire des agents conversationnels qui répondent aux entrées utilisateur

## 🏗️ Comment ça fonctionne dans Microsoft Agent Framework

Le framework fournit trois composants clés pour HITL :

1. **`RequestInfoExecutor`** - Un exécuteur spécial qui met en pause le workflow et émet un `RequestInfoEvent`
2. **`RequestInfoMessage`** - Classe de base pour les données de requête typées envoyées aux humains
3. **`RequestResponse`** - Corrèle les réponses humaines avec les requêtes originales via `request_id`

**Modèle de workflow :**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 Notre exemple : Réservation d'hôtel avec confirmation utilisateur

Nous allons développer le workflow conditionnel en ajoutant une confirmation humaine **avant** de suggérer des destinations alternatives :

1. L'utilisateur demande une destination (par exemple, "Paris")
2. `availability_agent` vérifie la disponibilité des chambres
3. **Si pas de chambres** → `confirmation_agent` demande "Souhaitez-vous voir des alternatives ?"
4. Le workflow **se met en pause** grâce à `RequestInfoExecutor`
5. **L'humain répond** "oui" ou "non" via la saisie console
6. `decision_manager` oriente selon la réponse :
   - **Oui** → Afficher des destinations alternatives
   - **Non** → Annuler la demande de réservation
7. Afficher le résultat final

Cela montre comment donner aux utilisateurs le contrôle sur les suggestions de l'agent !

---

Commençons ! 🚀


## Étape 1 : Importer les bibliothèques nécessaires

Nous importons les composants standards du cadre Agent ainsi que **les classes spécifiques à l'interaction humaine** :
- `RequestInfoExecutor` - Exécuteur qui met en pause le flux de travail pour une saisie humaine
- `RequestInfoEvent` - Événement émis lorsqu'une saisie humaine est requise
- `RequestInfoMessage` - Classe de base pour les charges utiles de requêtes typées
- `RequestResponse` - Corrèle les réponses humaines aux requêtes
- `WorkflowOutputEvent` - Événement pour détecter les sorties du flux de travail


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## Étape 2 : Définir les modèles Pydantic pour les sorties structurées

Ces modèles définissent le **schéma** que les agents retourneront. Nous conservons tous les modèles du flux conditionnel et ajoutons :

**Nouveautés pour l’humain dans la boucle :**
- `HumanFeedbackRequest` - Sous-classe de `RequestInfoMessage` qui définit la charge utile de la demande envoyée aux humains
  - Contient `prompt` (question à poser) et `destination` (contexte sur la ville indisponible)


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## Étape 3 : Créer l’Outil de Réservation d’Hôtel

Même outil que dans le flux de travail conditionnel - vérifie si des chambres sont disponibles à la destination.


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## Étape 4 : Définir les fonctions de condition pour le routage

Nous avons besoin de **quatre fonctions de condition** pour notre workflow humain dans la boucle :

**Du workflow conditionnel :**
1. `has_availability_condition` - Routage quand les hôtels SONT disponibles
2. `no_availability_condition` - Routage quand les hôtels NE SONT PAS disponibles

**Nouveau pour l'humain dans la boucle :**
3. `user_wants_alternatives_condition` - Routage quand l'utilisateur dit "oui" aux alternatives
4. `user_declines_alternatives_condition` - Routage quand l'utilisateur dit "non" aux alternatives


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## Étape 5 : Créez l'Executor du Gestionnaire de Décision

C'est le **cœur du modèle avec intervention humaine** ! Le `DecisionManager` est un `Executor` personnalisé qui :

1. **Reçoit des retours humains** via des objets `RequestResponse`
2. **Traite la décision de l'utilisateur** (oui/non)
3. **Dirige le flux de travail** en envoyant des messages aux agents appropriés

Fonctionnalités clés :
- Utilise le décorateur `@handler` pour exposer les méthodes en tant qu'étapes du flux de travail
- Reçoit `RequestResponse[HumanFeedbackRequest, str]` contenant à la fois la demande originale et la réponse de l'utilisateur
- Produit des messages simples "oui" ou "non" qui déclenchent nos fonctions conditionnelles


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## Étape 6 : Créer un exécuteur d'affichage personnalisé

Le même exécuteur d'affichage que celui du workflow conditionnel - produit les résultats finaux en tant que sortie du workflow.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## Étape 7 : Charger les variables d'environnement

Configurez le client LLM (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## Étape 8 : Créer des Agents et Exécuteurs IA

Nous créons **six composants de flux de travail** :

**Agents (emballés dans AgentExecutor) :**
1. **availability_agent** - Vérifie la disponibilité des hôtels avec l’outil
2. **confirmation_agent** - 🆕 Prépare la demande de confirmation humaine
3. **alternative_agent** - Suggère des villes alternatives (lorsque l’utilisateur dit oui)
4. **booking_agent** - Encourage la réservation (lorsque les chambres sont disponibles)
5. **cancellation_agent** - 🆕 Gère le message d’annulation (lorsque l’utilisateur dit non)

**Exécuteurs spéciaux :**
6. **request_info_executor** - 🆕 `RequestInfoExecutor` qui suspend le flux de travail pour la saisie humaine
7. **decision_manager** - 🆕 Exécuteur personnalisé qui oriente selon la réponse humaine (déjà défini ci-dessus)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## Étape 9 : Construire le workflow avec intervention humaine

Maintenant, nous construisons le graphe du workflow avec **acheminement conditionnel** incluant le chemin avec intervention humaine :

**Structure du workflow :**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**Arêtes clés :**
- `availability_agent → confirmation_agent` (quand aucune chambre)
- `confirmation_agent → prepare_human_request` (type de transformation)
- `prepare_human_request → request_info_executor` (pause pour humain)
- `request_info_executor → decision_manager` (toujours - fournit RequestResponse)
- `decision_manager → alternative_agent` (quand l’utilisateur dit "oui")
- `decision_manager → cancellation_agent` (quand l’utilisateur dit "non")
- `availability_agent → booking_agent` (quand chambres disponibles)
- Tous les chemins se terminent à `display_result`


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## Étape 10 : Exécuter le cas de test 1 - Ville SANS Disponibilité (Paris avec Confirmation Humaine)

Ce test démontre le **cycle complet avec intervention humaine** :

1. Demande d'hôtel à Paris
2. availability_agent vérifie → Pas de chambres
3. confirmation_agent crée une question à destination de l'humain
4. request_info_executor **met le workflow en pause** et émet `RequestInfoEvent`
5. **L'application détecte l'événement et invite l'utilisateur dans la console**
6. L'utilisateur tape "oui" ou "non"
7. L'application envoie la réponse via `send_responses_streaming()`
8. decision_manager oriente selon la réponse
9. Résultat final affiché

**Motif clé :**
- Utiliser `workflow.run_stream()` pour la première itération
- Utiliser `workflow.send_responses_streaming(pending_responses)` pour les itérations suivantes
- Écouter `RequestInfoEvent` pour détecter quand une entrée humaine est requise
- Écouter `WorkflowOutputEvent` pour capturer les résultats finaux


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## Étape 11 : Exécuter le cas de test 2 - Ville AVEC disponibilité (Stockholm - Aucune intervention humaine nécessaire)

Ce test démontre le **chemin direct** lorsque des chambres sont disponibles :

1. Demander un hôtel à Stockholm
2. availability_agent vérifie → Chambres disponibles ✅
3. booking_agent propose une réservation
4. display_result affiche la confirmation
5. **Aucune intervention humaine requise !**

Le flux de travail contourne entièrement le chemin avec intervention humaine lorsque des chambres sont disponibles.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## Key Takeaways and Human-in-the-Loop Best Practices

### ✅ What You've Learned:

#### 1. **RequestInfoExecutor Pattern**
The human-in-the-loop pattern in Microsoft Agent Framework uses three key components:
- `RequestInfoExecutor` - Pauses workflow and emits events
- `RequestInfoMessage` - Base class for typed request payloads (subclass this!)
- `RequestResponse` - Correlates human responses with original requests

**Critical Understanding:**
- `RequestInfoExecutor` does NOT collect input itself - it only pauses the workflow
- Your application code must listen for `RequestInfoEvent` and collect input
- You must call `send_responses_streaming()` with a dict mapping `request_id` to user's answer

#### 2. **Streaming Execution Pattern**
```python
# First iteration
stream = workflow.run_stream(initial_request)

# Subsequent iterations (after human input)
stream = workflow.send_responses_streaming(pending_responses)

# Always process events
events = [event async for event in stream]
```

#### 3. **Event-Driven Architecture**
Listen for specific events to control workflow:
- `RequestInfoEvent` - Human input is needed (workflow paused)
- `WorkflowOutputEvent` - Final result is available (workflow complete)
- `WorkflowStatusEvent` - State changes (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS, etc.)

#### 4. **Custom Executors with @handler**
The `DecisionManager` demonstrates how to create executors that:
- Use `@handler` decorator to expose methods as workflow steps
- Receive typed messages (e.g., `RequestResponse[HumanFeedbackRequest, str]`)
- Route workflow by sending messages to other executors
- Access context via `WorkflowContext`

#### 5. **Conditional Routing with Human Decisions**
You can create condition functions that evaluate human responses:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 Real-World Applications:

1. **Approval Workflows**
   - Get manager approval before processing expense reports
   - Require human review before sending automated emails
   - Confirm high-value transactions before execution

2. **Content Moderation**
   - Flag questionable content for human review
   - Ask moderators to make final decision on edge cases
   - Escalate to humans when AI confidence is low

3. **Customer Service**
   - Let AI handle routine questions automatically
   - Escalate complex issues to human agents
   - Ask customer if they want to speak to a human

4. **Data Processing**
   - Ask humans to resolve ambiguous data entries
   - Confirm AI interpretations of unclear documents
   - Let users choose between multiple valid interpretations

5. **Safety-Critical Systems**
   - Require human confirmation before irreversible actions
   - Get approval before accessing sensitive data
   - Confirm decisions in regulated industries (healthcare, finance)

6. **Interactive Agents**
   - Build conversational bots that ask follow-up questions
   - Create wizards that guide users through complex processes
   - Design agents that collaborate with humans step-by-step

### 🔄 Comparison: With vs Without Human-in-the-Loop

| Feature | Conditional Workflow | Human-in-the-Loop Workflow |
|---------|---------------------|---------------------------|
| **Execution** | Single `workflow.run()` | Loop with `run_stream()` + `send_responses_streaming()` |
| **User Input** | None (fully automated) | Interactive prompts via `input()` or UI |
| **Components** | Agents + Executors | + RequestInfoExecutor + DecisionManager |
| **Events** | AgentExecutorResponse only | RequestInfoEvent, WorkflowOutputEvent, etc. |
| **Pausing** | No pausing | Workflow pauses at RequestInfoExecutor |
| **Human Control** | No human control | Humans make key decisions |
| **Use Case** | Automation | Collaboration & oversight |

### 🚀 Advanced Patterns:

#### Multiple Human Decision Points
You can have multiple `RequestInfoExecutor` nodes in the same workflow:
```python
.add_edge(agent1, request_info_1)  # First human decision
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # Second human decision
.add_edge(decision_manager_2, final_agent)
```

#### Timeout Handling
Implement timeouts for human responses:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # Default to safe option
```

#### Rich UI Integration
Instead of `input()`, integrate with web UI, Slack, Teams, etc.:
```python
if isinstance(event, RequestInfoEvent):
    # Send notification to user's preferred channel
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### Conditional Human-in-the-Loop
Only ask for human input in specific situations:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # Only route to human if confidence is low or value is high
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ Best Practices:

1. **Always Subclass RequestInfoMessage**
   - Provides type safety and validation
   - Enables rich context for UI rendering
   - Clarifies intent of each request type

2. **Use Descriptive Prompts**
   - Include context about what you're asking
   - Explain consequences of each choice
   - Keep questions simple and clear

3. **Handle Unexpected Input**
   - Validate user responses
   - Provide defaults for invalid input
   - Give clear error messages

4. **Track Request IDs**
   - Use the correlation between request_id and responses
   - Don't try to manage state manually

5. **Design for Non-Blocking**
   - Don't block threads waiting for input
   - Use async patterns throughout
   - Support concurrent workflow instances

### 📚 Related Concepts:

- **Agent Middleware** - Intercept agent calls (previous notebook)
- **Workflow State Management** - Persist workflow state between runs
- **Multi-Agent Collaboration** - Combine human-in-the-loop with agent teams
- **Event-Driven Architectures** - Build reactive systems with events

---

### 🎓 Congratulations!

You've mastered human-in-the-loop workflows with Microsoft Agent Framework! You now know how to:
- ✅ Pause workflows to gather human input
- ✅ Use RequestInfoExecutor and RequestInfoMessage
- ✅ Handle streaming execution with events
- ✅ Create custom executors with @handler
- ✅ Route workflows based on human decisions
- ✅ Build interactive AI agents that collaborate with humans

**This is a critical pattern for building trustworthy, controllable AI systems!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
